In [1]:
import cv2
import os
import numpy as np
from sklearn.metrics import accuracy_score
from skimage.feature import hog
import random

In [2]:
def load_images_folder_histogram(directory):
# 49.28571428571429% in 3 iterations
    data = [] #Feature matrix
    labels = []
    # Loading of all sub-folders in the directory
    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)

    # Getting path of image files one-by-one from a sub-folder 'subdir_path'
        if os.path.isdir(subdir_path):
            for filename in os.listdir(subdir_path):
                filepath = os.path.join(subdir_path, filename) #Address/path of an image

                #Checking whether obtained path is a filepath
                if os.path.isfile(filepath):

                    # Load and preprocess image (resize to uniform size)
                    image = cv2.imread(filepath, cv2.IMREAD_COLOR)  # Convert to grayscale using argument as cv2.IMREAD_GRAYSCALE

                    image = cv2.resize(image, (64, 64))  # Resize to 64x64 pixels

                    arr = image.flatten() #flattening the image of size 64x64 to make one-dimentional array
                    arr1 = np.histogram(arr,256) #Computation of histogram of the array with 256 bins
                    #arr1 = [np.average(arr), np.max(arr), np.median(arr)]  #Other features

                    #data.append(image.flatten())  # Flatten the image

                    #Assigning suitable tables to different classes
                    data.append(arr1[0])
                    if subdir == 'dogs':
                        labels.append(0)  # May use the folder name as the label
                    else:
                        labels.append(1)

    return np.array(data), np.array(labels)  #Convering results to Numpy arrays

In [3]:
def load_images_folder_mmm(directory):
# 47.85714285714286% in 3 iterations

    data = [] #Feature matrix
    labels = []
    # Loading of all sub-folders in the directory
    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)

    # Getting path of image files one-by-one from a sub-folder 'subdir_path'
        if os.path.isdir(subdir_path):
            for filename in os.listdir(subdir_path):
                filepath = os.path.join(subdir_path, filename) #Address/path of an image

                #Checking whether obtained path is a filepath
                if os.path.isfile(filepath):

                    # Load and preprocess image (resize to uniform size)
                    image = cv2.imread(filepath, cv2.IMREAD_COLOR)  # Convert to grayscale using argument as cv2.IMREAD_GRAYSCALE

                    image = cv2.resize(image, (64, 64))  # Resize to 64x64 pixels

                    arr = image.flatten() #flattening the image of size 64x64 to make one-dimentional array
                    # arr1 = np.histogram(arr,256) #Computation of histogram of the array with 256 bins
                    arr1 = [np.average(arr), np.max(arr), np.median(arr)]  #Other features

                    #data.append(image.flatten())  # Flatten the image

                    #Assigning suitable tables to different classes
                    data.append(arr1)
                    if subdir == 'dogs':
                        labels.append(0)  # May use the folder name as the label
                    else:
                        labels.append(1)

    return np.array(data), np.array(labels)  #Convering results to Numpy arrays

In [4]:
def load_images_folder_gabor(directory):
    # 48.57142857142857% in 3 iterations
    data = []
    labels = []

    # Gabor filter parameters
    ksize = 5  # Kernel size
    sigma = 1.0
    lambd = 10.0
    gamma = 0.5
    thetas = [0, 45, 90, 135]  # Orientations in degrees

    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)

        if os.path.isdir(subdir_path):
            for filename in os.listdir(subdir_path):
                filepath = os.path.join(subdir_path, filename)

                if os.path.isfile(filepath):
                    # Load and preprocess image
                    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
                    image = cv2.resize(image, (64, 64))  # Resize to 64x64

                    gabor_features = []
                    for theta in thetas:
                        theta_rad = np.deg2rad(theta)
                        kernel = cv2.getGaborKernel((ksize, ksize), sigma, theta_rad, lambd, gamma, 0, ktype=cv2.CV_32F)
                        filtered_img = cv2.filter2D(image, cv2.CV_8UC3, kernel)  # Apply Gabor filter

                        gabor_features.append(np.mean(filtered_img))  # Mean response
                        gabor_features.append(np.var(filtered_img))   # Variance response

                    data.append(gabor_features)  # Store extracted features

                    # Assign class labels
                    labels.append(0 if subdir == 'dogs' else 1)

    return np.array(data), np.array(labels)  # Convert to NumPy arrays

In [5]:
def load_images_folder_hog(directory):
    # 49.28571428571429% in 3 iterations
    data = []
    labels = []

    for subdir in os.listdir(directory):
        subdir_path = os.path.join(directory, subdir)

        if os.path.isdir(subdir_path):
            for filename in os.listdir(subdir_path):
                filepath = os.path.join(subdir_path, filename)

                if os.path.isfile(filepath):
                    # Load and preprocess image
                    image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
                    image = cv2.resize(image, (64, 64))  # Resize to 64x64

                    # Extract HOG features
                    hog_features = hog(image, orientations=9, pixels_per_cell=(8, 8),
                                       cells_per_block=(2, 2), block_norm='L2-Hys', feature_vector=True)

                    data.append(hog_features)  # Store extracted HOG features

                    # Assign class labels
                    labels.append(0 if subdir == 'dogs' else 1)

    return np.array(data), np.array(labels)  # Convert to NumPy arrays

In [21]:
# Path to the dataset
dataset_directory1 = r"C:/D folder/Programming/programs/my codes/ML codes/knn classifier/Dog-Cat Dataset/train"
dataset_directory2 = r"C:/D folder/Programming/programs/my codes/ML codes/knn classifier/Dog-Cat Dataset/test"

# Load images and labels
images_train, labels_train = load_images_folder_hog(dataset_directory1) #images_train & images_test represent features of training and testing data, respect.
images_test, labels_test = load_images_folder_hog(dataset_directory2)

In [22]:
images_train.shape

(557, 1764)

In [23]:
def k_mean_Clustering(feature_train, k):
    r1 = []
    for i in range(0, k):
        n = random.randint(1, feature_train.shape[0])
        r1.append(n)
    rc = feature_train[r1]
    rc1 = np.zeros((rc.shape))
    cluster_no = np.zeros((feature_train.shape[0],1))
    iteration = 1

    while np.max(abs(np.array(rc)-np.array(rc1))):
        if iteration>1:
            rc = rc1
        for i in range((feature_train.shape[0])):
            dist = []
            for j in range(k):
                t = np.linalg.norm(np.array(feature_train[i]) - np.array(rc[j]))
                dist.append(t)
            p = np.argmin(dist)
            cluster_no[i] = p

        for l in range(k):
            data_c=[]
            for m in range((feature_train.shape[0])):
                if cluster_no[m] == l:
                    data_c.append(feature_train[m])
            if len(data_c) == 0:
                s = random.randint(1, feature_train.shape[0])
                rc1[l] = feature_train[s]
            else:
                rc1[l] = np.average(data_c,0)

        iteration = iteration+1
        if iteration > 50:
            break
    return np.array(rc1), np.array(cluster_no), iteration



In [24]:
k=2
centroids, cluster_n, iteration = k_mean_Clustering(images_train, 2)

In [25]:
predict_label = []
for i in range((images_test.shape[0])):
    dist = []
    for j in range(k):
        t = np.linalg.norm(np.array(images_test[i]) - np.array(centroids[j]))
        dist.append(t)
    predict_label.append(np.argmin(dist))

accuracy = accuracy_score(labels_test, predict_label)
print(f"Classification Accuracy: {100*accuracy}% in {iteration} iterations")

Classification Accuracy: 49.28571428571429% in 3 iterations
